# 02 - Model Architecture and Training
The key architectural change over standard ResNet-50 is replacing Global Average Pooling (GAP) with Log-Sum-Exp (LSE) Pooling. Two-Phase Training: Backbone frozen, then unfreezes at a low learning rate.

In [ ]:
# Cell 3: Model Architecture[cite: 1]
class LSEPooling(nn.Module):
    def __init__(self, r=5):
        super().__init__()
        self.r = r

    def forward(self, x):
        x_flat = x.view(x.size(0), x.size(1), -1)
        x_max, _ = torch.max(x_flat, dim=2, keepdim=True)
        lse = x_max + (1/self.r) * torch.log(
            torch.mean(torch.exp(self.r * (x_flat - x_max)), dim=2, keepdim=True) + 1e-8
        )
        return lse.view(x.size(0), x.size(1), 1, 1)

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.avgpool = LSEPooling(r=CFG["lse_r"])
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(CFG["device"])

labels = np.array(train_set.targets)
class_sample_count = np.array([len(np.where(labels == t)[0]) for t in np.unique(labels)])
weight = 1. / class_sample_count
weight = weight / weight.sum()
class_weights = torch.from_numpy(weight).float().to(CFG["device"])

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

print(f"Architecture: ResNet-50 with LSE-Pooling (r={CFG['lse_r']}) initialized.")
print("Status: Backbone frozen for Warmup Phase.")

In [ ]:
# Cell 4: Training[cite: 1]
def cutmix_data(x, y, alpha=0.4):
    x = x.clone()
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    W, H = x.size(2), x.size(3)
    cut_rat = np.sqrt(1. - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    
    x[:, :, x1:x2, y1:y2] = x[index, :, x1:x2, y1:y2]
    lam = 1 - ((x2 - x1) * (y2 - y1)) / (W * H)
    y_a, y_b = y, y[index]
    
    return x, y_a, y_b, lam

def train_model(model, criterion, optimizer, scheduler, num_epochs=15):
    best_model_wts = copy.deepcopy(model.state_dict())
    best_loss = float('inf')
    early_stop_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 10)
        
        if epoch == 2:
            print("[Phase 2] Unfreezing backbone for fine-tuning...")
            for param in model.parameters():
                param.requires_grad = True
            optimizer = optim.Adam(model.parameters(), lr=CFG["lr_fine_tune"])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=(num_epochs - 2)
            )

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = train_loader
            else:
                model.eval()
                dataloader = val_loader

            running_loss = 0.0
            optimizer.zero_grad()
            
            for batch_idx, (inputs, labels) in enumerate(tqdm(dataloader, desc=f"{phase.upper()}", leave=False)):
                inputs, labels = inputs.to(CFG["device"]), labels.to(CFG["device"])

                with torch.set_grad_enabled(phase == 'train'):
                    if phase == 'train' and np.random.random() > 0.75:
                        inputs, y_a, y_b, lam = cutmix_data(inputs, labels)
                        outputs = model(inputs)
                        loss = lam * criterion(outputs, y_a) + (1 - lam) * criterion(outputs, y_b)
                    else:
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)

                    running_loss += loss.item() * inputs.size(0)

                    if phase == 'train':
                        scaled_loss = loss / CFG["accumulation_steps"]
                        scaled_loss.backward()
                        
                        if ((batch_idx + 1) % CFG["accumulation_steps"] == 0) or (batch_idx + 1 == len(dataloader)):
                            optimizer.step()
                            optimizer.zero_grad()

            epoch_loss = running_loss / len(dataloader.dataset)
            print(f'{phase.upper()} Loss: {epoch_loss:.4f}')

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                if scheduler is not None:
                    scheduler.step()
                    print(f"Current Learning Rate: {scheduler.get_last_lr()[0]:.6f}")
                else:
                    print(f"Warmup Learning Rate (Static): {optimizer.param_groups[0]['lr']:.6f}")
            else:
                history['val_loss'].append(epoch_loss)
                if epoch_loss < best_loss:
                    best_loss = epoch_loss
                    best_model_wts = copy.deepcopy(model.state_dict())
                    torch.save(model.state_dict(), CFG["model_path"])
                    early_stop_counter = 0
                    print("--> Validation improved. Model saved.")
                else:
                    early_stop_counter += 1

        if early_stop_counter >= CFG["patience"]:
            print(f"\nEarly stopping triggered at epoch {epoch}")
            break
            
    model.load_state_dict(best_model_wts)
    return model, history

optimizer = optim.Adam(model.fc.parameters(), lr=CFG["lr_warmup"])
scheduler = None

trained_model, loss_history = train_model(
    model, criterion, optimizer, scheduler, num_epochs=CFG["epochs"]
)

In [ ]:
# Cell 5: Training Visualization[cite: 1]
def plot_training_history(history):
    if not history or not history['train_loss']:
        print("Error: No training history found.")
        return

    plt.figure(figsize=(10, 6))
    epochs = range(len(history['train_loss']))
    
    plt.plot(epochs, history['train_loss'], label='Training Loss (with CutMix/Aug)', color='#4C72B0', lw=2.5, marker='o', markersize=5)
    plt.plot(epochs, history['val_loss'], label='Validation Loss (Clean Data)', color='#C44E52', lw=2.5, marker='s', markersize=5)
    
    if len(epochs) > 2:
        plt.axvline(x=2, color='gray', linestyle='--', alpha=0.8)
        y_max = max(history['train_loss'])
        plt.text(2.2, y_max * 0.9, 'Phase 2: Backbone Unfrozen', color='dimgray', fontweight='bold', fontsize=11)

    plt.title('Training and Validation Loss: Optimization Progress', fontsize=15, fontweight='bold', pad=15)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss (Cross Entropy)', fontsize=12)
    plt.legend(loc='upper right', frameon=True, shadow=True)
    plt.grid(True, alpha=0.3, linestyle='--')
    sns.despine()
    plt.tight_layout()
    plt.show()

plot_training_history(loss_history)